<a href="https://colab.research.google.com/github/adithi2307/IN126055302_GEN_AI/blob/main/TASK_3_CHATBOT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
import os

os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
os.environ["DISABLE_TQDM"] = "1"
from transformers.utils import logging
logging.set_verbosity_error()

from transformers import AutoTokenizer, AutoModelForCausalLM
from transformers.utils import logging
logging.set_verbosity_error()
import logging
from transformers import logging as transformers_logging
transformers_logging.set_verbosity_error()

!pip install transformers torch --quiet

In [7]:
# Standard library imports
import warnings
warnings.filterwarnings('ignore')
import torch

model_name = "Qwen/Qwen2.5-1.5B-Instruct"
# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name)
# Load model with automatic dtype and device placement
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype="auto",
    device_map="auto"
)
model.eval()
print("Model loaded successfully.")
print("Device map:", model.hf_device_map if hasattr(model, "hf_device_map") else "Single device")

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Model loaded successfully.
Device map: Single device


In [8]:
def generate_reply(chat_history, user_message, max_new_tokens=256):

    chat_history.append({"role": "user", "content": user_message})

    prompt_text = tokenizer.apply_chat_template(
        chat_history,
        tokenize=False,
        add_generation_prompt=True,
    )

    inputs = tokenizer(
        [prompt_text],
        return_tensors="pt"
    ).to(model.device)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            repetition_penalty=1.05
        )


    generated_ids = output_ids[0, inputs["input_ids"].shape[1]:]
    assistant_reply = tokenizer.decode(
        generated_ids,
        skip_special_tokens=True
    ).strip()

    chat_history.append({"role": "assistant", "content": assistant_reply})
    return chat_history, assistant_reply

In [5]:
def run_chatbot():
    print("Chatbot: Hello! I am your AI assistant. How can I help you today?")
    # Initialize
    chat_history = [
        {
            "role": "system",
            "content": "You are a helpful, concise AI assistant. Answer clearly and politely."
        }
    ]
    while True:
        user_input = input("User: ").strip()

        # Exit condition
        if user_input.lower() in ["exit", "quit"]:
            print("Chatbot: Goodbye! Have a great day.")
            break

        # Skip empty input
        if not user_input:
            print("Chatbot: Please type something so I can respond.")
            continue

        chat_history, assistant_reply = generate_reply(chat_history, user_input)
        if assistant_reply == "":
            assistant_reply = "I'm not sure how to respond to that yet, but I'm learning."
        print("Chatbot:", assistant_reply)
run_chatbot()

Chatbot: Hello! I am your AI assistant. How can I help you today?
User: hello
Chatbot: Hello! How can I assist you today?
User: what is artificial intelligence?
Chatbot: Artificial Intelligence (AI) refers to the simulation of human intelligence in machines that are programmed to think and learn like humans. It involves the use of algorithms and statistical models to perform tasks that typically require human intelligence, such as visual perception, speech recognition, decision-making, and language translation. AI systems can be designed to adapt and improve their performance over time based on new data or experiences.
User: who created python?
Chatbot: Python was created by Guido van Rossum in 1989.
User: thank you
Chatbot: You're welcome! If you have any other questions, feel free to ask.
User: exit
Chatbot: Goodbye! Have a great day.
